In [1]:
import os
# from dotenv import load_dotenv
# load_dotenv(r"D:\GenAI\.env", override=True)

os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

In [2]:
from langchain.chat_models import init_chat_model

### OpenAI Model

#####  Dont have credits -> ERROR

In [3]:
model = init_chat_model("gpt-4.1")

# response=model.invoke("Hello How are you?")
# response.content

In [4]:
from langchain_openai import ChatOpenAI
model= ChatOpenAI(model="gpt-4.1")

# response = model.invoke("Hello How are you?")
# response.content

### Google Gemini Model

In [5]:
model = init_chat_model("google_genai:gemini-3.1-flash-lite")
response = model.invoke("Why do parrots talk?")
response.content

GoogleAPIError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

In [6]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite")
response = model.invoke("Why do parrots talk?")
response.content

GoogleAPIError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

### Groq Model


In [7]:
from langchain.chat_models import init_chat_model

model = init_chat_model("groq:openai/gpt-oss-120b")

response = model.invoke("Why do parrots talk?")
response.content

"Parrots aren’t really “talking” in the way humans do, but many species are excellent mimics and can reproduce human speech sounds. Their ability to do so is the result of a combination of anatomy, brain structure, and social behavior. Here’s a breakdown of the main reasons:\n\n---\n\n## 1. **Vocal anatomy that’s built for mimicry**\n\n| Feature | What it does | Why it matters for speech |\n|---------|--------------|---------------------------|\n| **Syrinx** (the bird’s voice box) | Located at the base of the trachea, the syrinx contains two sets of vibrating membranes that can be controlled independently. | Allows parrots to produce a huge variety of sounds, including rapid changes in pitch and tone that are essential for imitating human speech. |\n| **Highly flexible tongue and beak** | Parrots can shape their beak and move their tongue in fine, precise ways. | Helps them match the articulation of human consonants and vowels. |\n| **Well‑developed respiratory control** | They can reg

In [8]:
from langchain_groq import ChatGroq

model = ChatGroq(model = "openai/gpt-oss-120b")

response = model.invoke("Why do parrots talk?")
response.content

'Parrots don’t “talk” in the way humans use language; they are exceptionally good at **vocal mimicry**, and the reasons they can imitate human speech (and other sounds) stem from a mix of anatomy, social behavior, and evolutionary adaptations.\n\n### 1. Anatomy that enables mimicry\n| Feature | How it helps |\n|---------|--------------|\n| **Syrinx** (bird’s voice box) | Located at the base of the trachea, the syrinx has highly flexible membranes and muscles that can produce a wide range of tones, far more varied than a human larynx. |\n| **Brain structures** | Parrots have a well‑developed **song system** (including nuclei like HVC and RA) and a large **caudomedial nidopallium**, which is involved in vocal learning and memory. |\n| **Tongue and beak control** | Precise, rapid movements let them shape sounds with fine detail. |\n\n### 2. Evolutionary and social motivations\n1. **Flocking communication** – In the wild, parrots live in noisy, dense flocks. Being able to copy and blend wi

## Streaming and Batch Inference

### Streaming 

In GenAI, streaming is the response-delivery pattern where the model emits each decoded token to the client immediately as it's generated, rather than buffering the full response before sending it. 

#### How it works mechanically:

1. The LLM generates tokens autoregressively (one at a time).
2. Instead of collecting all tokens into a single HTTP body, the server opens a persistent connection — typically Server-Sent Events (SSE), HTTP chunked transfer, or WebSocket — and pushes each token (or small group of tokens) the moment it's ready. 
3. The client appends each chunk to the UI, producing the familiar "typewriter" effect.

This improves user experience for long outputs. 

#### Use model.stream()

In [9]:
model.invoke("Write me a 200 words paragraph on Artificial Intelligence")

AIMessage(content='Artificial Intelligence (AI) is a multidisciplinary field that seeks to create systems capable of performing tasks traditionally requiring human cognition, such as learning, reasoning, perception, and decision‑making. By leveraging algorithms inspired by neural networks, statistical inference, and evolutionary processes, AI can analyze massive datasets, recognize patterns, and generate predictions with a speed and accuracy far beyond human capability. Modern AI applications span diverse domains: autonomous vehicles interpret sensor data to navigate complex environments; medical diagnostics employ deep‑learning models to detect diseases from imaging with remarkable precision; natural‑language processors like chatbots and translation services understand and generate human‑like text, breaking language barriers. Despite these advances, AI also raises profound ethical and societal challenges, including bias amplification, privacy intrusion, job displacement, and the poten

In [10]:
model.stream("Write me a 200 words paragraph on Artificial Intelligence")

<generator object BaseChatModel.stream at 0x000001221D486840>

In [11]:
for chunk in model.stream("Write me a 200-word paragraph on Artificial Intelligence."):
    if chunk.text:
        print(chunk.text, end="|", flush=True)

Artificial| intelligence| (|AI|)| represents| a| transformative| convergence| of| computer| science|,| mathematics|,| and| cognitive| theory|,| enabling| machines| to| perform| tasks| that| traditionally| required| human| intelligence| such| as| perception|,| reasoning|,| learning|,| and| decision|‑|making|.| Modern| AI| systems| rely| heavily| on| machine|‑|learning| algorithms|,| particularly| deep| neural| networks|,| which| extract| hierarchical| patterns| from| massive| datasets|,| allowing| applications| ranging| from| image| and| speech| recognition| to| natural|‑|language| generation| and| autonomous| navigation|.| Beyond| technical| prowess|,| AI| resh|apes| economic| structures| by| autom|ating| routine| labor|,| augment|ing| creative| processes|,| and| fostering| new| industries| built| around| data| analytics|,| personalized| medicine|,| and| intelligent| robotics|.| Ethical| considerations| accompany| this| rapid| advancement|:| bias| embedded| in| training| data| can| per

## Batch Inference

In GenAI, batch (or batch inference) is the counterpart to streaming: instead of processing requests one-by-one in real time, you group many requests together and process them as a single job, typically asynchronously.

#### How it works:

You submit a large set of prompts (e.g., a JSONL file of up to 50,000 requests to OpenAI's Batch API) as one job.
The provider processes them during off-peak compute windows, returning results within a guaranteed SLA (usually 24 hours, often much faster in practice). 
You poll for completion and retrieve the full output file. 

#### Use model.stream()


In [12]:
responses = model.batch(
    [
        "Why do parrots have colorful feather?",
        "How do airplanes fly?",
        "What is quantum computing?"
    ],
    config={
        'max_concurrency' : 5, # Limit to 5 parallel calls.
    }
)

responses

[AIMessage(content='Parrots are famous for their brilliant, rainbow‑like plumage, and the colors you see are the result of a mix of biology, physics, and evolution. Here’s a breakdown of why their feathers are so colorful:\n\n---\n\n## 1. The chemistry behind the pigments  \n\n### **a. Carotenoids**\n* **What they are:** Fat‑soluble pigments that birds obtain from their diet (e.g., fruits, seeds, nuts, and some insects).  \n* **Colors produced:** Reds, oranges, and yellows.  \n* **Why it matters:** Because parrots eat a lot of fruit and seed material rich in carotenoids, these pigments get deposited in the feather keratin, giving many species their vivid orange‑red tones.\n\n### **b. Psittacofulvins**\n* **What they are:** A unique group of pigments *only* found in parrots (the name literally means “parrot pigments”).  \n* **Colors produced:** Bright reds, oranges, and yellows that are *not* derived from diet.  \n* **Why it matters:** Even if a parrot’s diet is low in carotenoids, it c